In [ ]:
import torch
import torchvision
import torch.nn as nn
import cv2
import random
import json
from typing import Literal
from pydantic import BaseModel
from pathlib import Path
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from  torchvision import models 
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
def image_to_tensor(img: np.ndarray, resize: tuple[int, int] | None = None, with_batch_size: bool = True) -> torch.Tensor:
    if resize is not None:
        img = cv2.resize(img, resize)

    if img.ndim == 2:
        img = img[:, :, np.newaxis]

    img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
    return img_tensor.unsqueeze(0) if with_batch_size else img_tensor


def tensor_to_image(img_tensor: torch.Tensor) -> np.ndarray:
    if img_tensor.shape[0] > 1:
        raise ValueError("batch size must be equals to 1")
    
    img_arr = img_tensor.detach().numpy()
    return img_arr.squeeze()


def prepare_img_for_resnet(img: np.ndarray):
    img = cv2.resize(img, (224, 224)).astype(np.float32)
    img /= 255
    img_tensor = torch.tensor(img).permute(2, 0, 1).float()# .unsqueeze(0)
    return torchvision.transforms.functional.normalize(img_tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
model = models.resnet18()

model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 1),
)

state = torch.load('/home/polymorvic/development/deep-peep-snooker/models/v5/shot-classifier-best.pt')
model.load_state_dict(state['model'])
model = model.to(device)
model.eval()

In [ ]:
img = cv2.imread('/home/polymorvic/development/deep-peep-snooker/10_002.png').astype(np.float32)  # skip_test
img = prepare_img_for_resnet(img)
img = img.unsqueeze(0)
img = img.to(device)

with torch.no_grad():
    y_hat = model(img)
    probs = F.sigmoid(y_hat)


In [ ]:
probs

In [ ]:
with open("split_spec.json", encoding="utf-8") as f:
    split_spec = json.load(f)

In [ ]:
import random

In [ ]:
val_pics = []
for _ in range(16):

    while True:
        pic = random.choice(split_spec['val'])
        if pic not in val_pics:
            val_pics.append(pic)
            break

In [ ]:
df = pd.read_csv('shot_labels.csv')

In [ ]:
val_pics

In [ ]:
val_df = df[df['img_name'].isin(val_pics)]

In [ ]:
val_df

In [ ]:
images = []
labels = []
preds = []
for _, row in val_df.iterrows():
    img_name = row['img_name']
    dir = 'pics/' if 'skip_' not in img_name else 'pics/skip/'
    image = cv2.imread(f'../{dir}{img_name}.png')
    images.append(image)

    labels.append(row['label'])


    image = prepare_img_for_resnet(image)
    image = image.unsqueeze(0)
    image = image.to(device)

    with torch.no_grad():
        y_hat = model(image)
        # print(y_hat)
        prob = F.sigmoid(y_hat)
        print(prob)

    preds.append(prob)

    

In [ ]:
val_df

In [ ]:
# testowe video

cap = cv2.VideoCapture('/home/polymorvic/Videos/Screencasts/Screencast from 2026-02-11 08-21-54.webm')

In [ ]:
cap.isOpened()

In [ ]:
cap.get(cv2.CAP_PROP_FRAME_COUNT)

In [ ]:
cap.get(cv2.CAP_PROP_FPS)

In [ ]:
cap = cv2.VideoCapture('/home/polymorvic/Videos/Screencasts/Screencast from 2026-02-11 08-21-54.webm')
counter = 0
while True:
    ret, frame = cap.read()

    if not ret:
        print('error')
        break

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # if 1000 < counter < 1020:
    #     plt.imshow(frame)
    #     plt.show()

    # counter +=1


    img = prepare_img_for_resnet(frame)
    img = img.unsqueeze(0)
    img = img.to(device)

    with torch.no_grad():
        y_hat = model(img)
        probs = F.sigmoid(y_hat).item()


    if probs >= 0.5:
        plt.imshow(frame)
        plt.show()



    # if counter > 30:
    #     break

In [ ]:
cap = cv2.VideoCapture('/home/polymorvic/Videos/Screencasts/Screencast from 2026-02-11 08-21-54.webm')
counter = 0
while True:
    ret, frame = cap.read()

    if not ret:
        print('error')
        break

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


    img = prepare_img_for_resnet(frame)
    img = img.unsqueeze(0)
    img = img.to(device)

    with torch.no_grad():
        y_hat = model(img)
        probs = F.sigmoid(y_hat).item()


    if probs >= 0.5:
        plt.imshow(frame)
        plt.show()



    # if counter > 30:
    #     break